# Table of Contents

1. [Human-in-the-Loop with Interrupts](#human-in-the-loop-with-interrupts)

   - What are Interrupts?

   - Why Use Interrupts?

   - Two Types: `interrupt_before` vs `interrupt_after`

   - How It Works (Execution Flow)

   - Basic Example

   - Key Concepts

   - Rules of Interrupts

     - Don't wrap `interrupt()` in Try/Except

     - Keep interrupt order consistent

     - Only JSON-serializable values

     - Side effects before interrupts run twice

   - When to Use / Summary

2. [Command — Dynamic & Conditional Interrupts](#command)

   - Command vs Regular Interrupts (static vs runtime)

   - Command Parameters (`update`, `goto`)

   - Example: Conditional Transaction Approval

   - Benefits & When to Use

3. [Code — Approval Workflow with `Command` + `interrupt()`](#code)

   - LLM Setup

   - Approval graph with `Command` routing (`approved` / `rejected`)

   - Interactive UI with `ipywidgets` for in-process decision

4. [External Approvals via Email / REST API](#external-approvals)

   - `request_approval` node — full lifecycle diagram

     - PENDING: graph pauses at `interrupt()`, state saved to DB

     - APPROVED / REJECTED: `Command(resume=decision)` resumes graph

   - State at each stage (`state.next`, `state.interrupts`, `state.values`)

   - Option 1 — Email Approval

     - `request_approval` node with token generation

     - `POST /workflows` — starts graph, sends approval email

     - `GET /approve?token=&decision=` — email link handler, resumes graph

   - Option 2 — REST API Approval

     - `POST /workflows` — starts graph, returns `task_id`

     - `POST /workflows/{task_id}/approve` — submits decision, resumes graph

     - `GET /workflows/{task_id}` — poll status

   - REST API caller flow (3-step sequence)

   - Key points: `interrupt()`, `Command(resume=x)`, `PostgresSaver`

   - Comparison: Email vs REST API

# Human-in-the-Loop with Interrupts

## What are Interrupts?

**Interrupts** pause LangGraph execution and wait for human input before continuing.

In [ ]:
**Visual Flow:**
START → Node A → [INTERRUPT] → Node B → END
                     ⏸️ Pause here for human review

---

## Why Use Interrupts?

| **Use Case** | **Example** |

|--------------|-------------|

| **Approval Workflows** | AI suggests deleting data → Human confirms |

| **Quality Control** | AI generates content → Human edits |

| **High-Stakes Actions** | Financial transactions → Human approves |

| **Multi-Step Processes** | Data pipeline → Pause between stages |

---

## Two Types of Interrupts

In [ ]:
### 1. `interrupt_before` - Pause BEFORE node execution
graph = workflow.compile(
    checkpointer=checkpointer,
    interrupt_before=["approval_node"]
)

In [ ]:
### 2. `interrupt_after` - Pause AFTER node execution
graph = workflow.compile(
    checkpointer=checkpointer,
    interrupt_after=["data_processing"]
)

---

## How It Works

### Execution Flow:

1. Graph executes nodes sequentially

2. Reaches interrupt point → ⏸️ **PAUSES**

3. Saves state to checkpoint

4. Returns control to you

5. You review/modify state using `update_state()`

6. Call `invoke(None, config)` → ▶️ **RESUMES**

7. Completes remaining nodes

In [ ]:
### During Pause:
# Check state
state = graph.get_state(config)

# Modify state
graph.update_state(config, {"approved": True})

# Resume
result = graph.invoke(None, config)

---

## Basic Example

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from typing_extensions import TypedDict

class State(TypedDict):
    action: str
    approved: bool

def suggest_action(state: State):
    return {"action": "Delete all data"}

def execute_action(state: State):
    return {"action": f"Executed: {state['action']}" if state["approved"] else "Cancelled"}

# Build graph
workflow = StateGraph(State)
workflow.add_node("suggest", suggest_action)
workflow.add_node("execute", execute_action)
workflow.add_edge(START, "suggest")
workflow.add_edge("suggest", "execute")
workflow.add_edge("execute", END)

# Compile WITH interrupt
graph = workflow.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["execute"]  # ⏸️ Pause here
)

# Execute
config = {"configurable": {"thread_id": "1"}}
result = graph.invoke({"action": "", "approved": False}, config)
# Graph PAUSED before "execute"

# Human approves
graph.update_state(config, {"approved": True})

# Resume
result = graph.invoke(None, config)  # Completes execution

---

## Key Concepts

| **Concept** | **Explanation** |

|-------------|----------------|

| **Interrupt Point** | Where graph pauses (before/after node) |

| **Checkpoint** | Saved state at interrupt |

| **Resume** | `invoke(None, config)` continues execution |

| **State Modification** | `update_state()` during pause |

| **Thread ID** | Required for checkpointing |

---

## Rules of Interrupts

In [ ]:
### 1. Don't Wrap `interrupt()` in Try/Except
❌ **Wrong:**
try:
    decision = interrupt("Approve?")  # Gets caught!
except:
    pass

In [ ]:
✅ **Correct:**
data = validate_data(state)  # Error handling separate
decision = interrupt("Approve?")  # Don't catch this

**Why:** `Interrupts use exceptions internally to pause execution.`

---

In [ ]:
### 2. Keep Interrupt Order Consistent
❌ **Wrong:**
if random_condition():
    interrupt("First?")  # Order changes on resume!
decision = interrupt("Second?")

In [ ]:
✅ **Correct:**
first = interrupt("First?")  # Always same order
second = interrupt("Second?")

**Why:** Resume values match interrupts by index position.

---

In [ ]:
### 3. Only JSON-Serializable Values
❌ **Wrong:**
interrupt(some_function)  # Can't serialize
interrupt(class_instance)  # Can't serialize

In [ ]:
✅ **Correct:**
interrupt("Approve?")  # String ✓
interrupt({"amount": 100})  # Dict ✓

**Why:** Checkpointers must serialize/deserialize values.

---

In [ ]:
### 4. Side Effects Before Interrupts Run Twice
❌ **Wrong:**
def node(state):
    db.insert(record)  # Runs AGAIN on resume!
    decision = interrupt("Approve?")

In [ ]:
✅ **Correct:**
def node(state):
    decision = interrupt("Approve?")
    if decision:
        db.insert(record)  # Runs once after resume

**Why:** Nodes re-execute from the beginning on resume.

---

## When to Use

✅ **Use When:**

- Irreversible actions (delete, send, deploy)

- Human judgment needed

- Compliance/approval required

❌ **Don't Use When:**

- Fully automated workflows

- Real-time responses required

- Low-risk operations

---

## Summary

| **Feature** | **Details** |

|-------------|-------------|

| **Purpose** | Pause execution for human review |

| **Requirements** | Checkpointer + thread_id |

| **Resume** | `invoke(None, config)` |

| **Benefits** | Safety, control, compliance |

---

## What is `Command`?

**`Command`** lets you **control graph execution from inside a node** - including conditional interrupts and dynamic routing.

---

## Command vs Regular Interrupts

| **Type** | **When Defined** | **Behavior** |

|----------|-----------------|--------------|

| `interrupt_before/after` | Compile time | Always pauses at that node |

| `Command` with `interrupt()` | Runtime | Pause only if condition met |

In [ ]:
### Static Interrupt (Always Pauses)
graph = workflow.compile(
    interrupt_before=["approval"]  # Always interrupts
)

In [ ]:
### Dynamic Interrupt (Conditional)
from langgraph.types import Command, interrupt

def smart_approval(state: State) -> Command:
    if state["amount"] > 1000:
        # Interrupt ONLY for large amounts
        decision = interrupt("Approve $" + str(state["amount"]) + "?")
        return Command(update={"approved": decision})

    # Auto-approve small amounts - NO interrupt
    return Command(update={"approved": True})

---

## Command Parameters

In [ ]:
Command(
    update={"field": "value"},      # Update state
    goto="next_node"                # Jump to specific node
)

---

## Example: Conditional Transaction Approval

In [ ]:
def validate_transaction(state) -> Command:
    if state["amount"] < 1000:
        # Small: auto-approve
        return Command(update={"approved": True})
    else:
        # Large: ask human
        decision = interrupt(f"Approve ${state['amount']}?")
        return Command(update={"approved": decision})

---

## Benefits

✅ **Conditional** - Interrupt only when needed

✅ **Efficient** - Don't bother humans for trivial cases

✅ **Dynamic routing** - Jump to different nodes

✅ **Better UX** - Smart automation + human oversight

---

## When to Use

- **Use `Command`**: Need conditional interrupts or dynamic routing

- **Use `interrupt_before/after`**: Always want to pause at a specific node

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
import os

load_dotenv()

api_key = os.environ['UNIFIED_LLM_KEY']
# print(api_key)
base_url = ""

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=256,
    api_key=api_key,
    base_url=base_url
)

In [ ]:
from typing import Literal, Optional, TypedDict
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, interrupt

class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal['pending', 'approved', 'rejected']]


def approval_request(state: ApprovalState) -> Command[Literal['approved', 'rejected']]:
    decision = interrupt({
        "question": "Approve this action?",
        "type": "radio",
        "options": ["Approve", "Reject"],
        "details": state['action_details']
    })

    return Command(goto="approved" if decision else 'rejected')

def proceed(state: ApprovalState) -> ApprovalState:
    return {"status": 'approved'}

def cancel(state: ApprovalState) -> ApprovalState:
    return {"status": 'rejected'}

workflow = StateGraph(ApprovalState)
workflow.add_node('approval_request', approval_request)
workflow.add_node('approved', proceed)
workflow.add_node('rejected', cancel)

workflow.add_edge(START, 'approval_request')
workflow.add_edge('approved', END)
workflow.add_edge('rejected', END)

checkpointer = MemorySaver()
graph = workflow.compile(checkpointer=checkpointer)

config = {"configurable": {"thread_id": "approval-123"}}
initial = graph.invoke(
    {"action_details": "Transfer $500", "status": "pending"},
    config=config,
)

print(initial["__interrupt__"])

# Get the interrupt data
state = graph.get_state(config)
print(f"\nAction to approve: {state.values['action_details']}")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Create radio button widget
radio = widgets.RadioButtons(
    options=['Approve', 'Reject'],
    description='Decision:',
    disabled=False
)

# Create submit button
button = widgets.Button(description="Submit Decision")
output = widgets.Output()

def on_button_click(b):
    with output:
        output.clear_output()
        decision = "approved" if radio.value == "Approve" else "rejected"
        print(f"✅ You selected: {radio.value}")

        # Resume graph with decision
        result = graph.invoke(
            Command(update={"status": decision}),
            config
        )

        print(f"\n🎯 Final Status: {result['status']}")
        print(f"Action: {result['action_details']}")

button.on_click(on_button_click)

# Display widgets
display(radio, button, output)

---

# External Approvals via Email / REST API

When approval can't happen in the same process (e.g., a manager reviews hours later), the graph **pauses at `interrupt()`**, your server returns a response, and the approval arrives later via an email link click or a REST API call.

---

## The `request_approval` Node — Full Lifecycle

```
graph.invoke() called
       │
       ▼
  [request_approval node runs]
       │
       ├─► interrupt() called ──────────────────────────────────────────────┐
       │   Graph PAUSES here                                                │
       │   State saved to PostgreSQL checkpointer                          │
       │   Control returned to your API                                     │
       │                                                                    │
       │   STATUS: PENDING  ──► state.next = ('request_approval',)         │
       │                        state.values["status"] = "pending"         │
       │                                                                    │
       │   Your API:                                                        │
       │     - Sends approval email  OR  returns task_id to caller         │
       │     - Saves (token → thread_id + checkpoint_id) in DB             │
       │                                                                    │
       │   ┌──── Hours/Days later ────────────────────────────────────┐    │
       │   │  Manager clicks email link  OR  caller POSTs /approve    │    │
       │   │  Your API fetches thread_id + checkpoint_id from DB      │    │
       │   │  Calls update_state(config, {"approved": True/False})    │    │
       │   │  Calls invoke(None, config)  ──► graph RESUMES           │    │
       │   └─────────────────────────────────────────────────────────-┘    │
       │                                                                    │
       ◄───────────────────────────────────────────────────────────────────┘
       │   interrupt() returns the decision value
       │
       ├─► decision == "approved"
       │     Command(goto="approved_node")
       │     STATUS: APPROVED  ──► state.values["status"] = "approved"
       │
       └─► decision == "rejected"
             Command(goto="rejected_node")
             STATUS: REJECTED  ──► state.values["status"] = "rejected"
```

### State at Each Stage

In [ ]:
# STAGE 1 — Before invoke (initial state)
{"action": "Deploy to prod", "amount": 5000, "status": "pending", "approved": False}

# STAGE 2 — After invoke hits interrupt (PAUSED)
state = graph.get_state(config)
state.next           # ('request_approval',)  ← non-empty = paused
state.values         # {"action": "Deploy to prod", "status": "pending", ...}
state.interrupts     # [Interrupt(value={"approval_token": "...", "action": ...})]

# STAGE 3 — After update_state + invoke (COMPLETED)
state.next           # ()  ← empty = completed
state.values         # {"action": "Deploy to prod", "status": "approved", ...}

---

## Option 1 — Email Approval

In [ ]:
from langgraph.types import Command, interrupt
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.postgres import PostgresSaver
from typing import TypedDict, Optional, Literal
from fastapi import FastAPI
import uuid, smtplib

# ── State ─────────────────────────────────────────────────────────
class ApprovalState(TypedDict):
    action:  str
    amount:  float
    status:  Literal["pending", "approved", "rejected"]
    token:   Optional[str]

# ── request_approval node ─────────────────────────────────────────
def request_approval(state: ApprovalState) -> Command:
    token = str(uuid.uuid4())            # unique token ties email link → checkpoint

    # ⏸️ Graph PAUSES here — interrupt() suspends execution
    # Whatever you pass to interrupt() is exposed via state.interrupts
    decision = interrupt({
        "approval_token": token,
        "action":         state["action"],
        "amount":         state["amount"]
    })
    # ▶️ Resumes here when invoke(None, config) is called after update_state()
    # decision = whatever was passed to update_state() — "approved" or "rejected"

    if decision == "approved":
        return Command(
            update={"status": "approved", "token": token},
            goto="on_approved"
        )
    return Command(
        update={"status": "rejected", "token": token},
        goto="on_rejected"
    )

def on_approved(state: ApprovalState):
    print(f"✅ Executing approved action: {state['action']}")
    return {"status": "approved"}

def on_rejected(state: ApprovalState):
    print(f"❌ Action rejected: {state['action']}")
    return {"status": "rejected"}

# ── Build graph ───────────────────────────────────────────────────
workflow = StateGraph(ApprovalState)
workflow.add_node("request_approval", request_approval)
workflow.add_node("on_approved",      on_approved)
workflow.add_node("on_rejected",      on_rejected)
workflow.add_edge(START, "request_approval")
workflow.add_edge("on_approved", END)
workflow.add_edge("on_rejected", END)

checkpointer = PostgresSaver.from_conn_string("postgresql://...")
graph = workflow.compile(checkpointer=checkpointer)

In [ ]:
# ── FastAPI: Start workflow ───────────────────────────────────────
app = FastAPI()

@app.post("/workflows")
async def start_workflow(user_id: str, action: str, amount: float):
    thread_id = f"user:{user_id}:{uuid.uuid4()}"
    config    = {"configurable": {"thread_id": thread_id}}

    # Run graph — pauses at interrupt(), returns immediately
    graph.invoke({"action": action, "amount": amount,
                  "status": "pending", "token": ""}, config)

    # Graph is now PAUSED — retrieve interrupt data
    state         = graph.get_state(config)
    interrupt_val = state.interrupts[0].value        # data passed to interrupt()
    token         = interrupt_val["approval_token"]
    checkpoint_id = state.config["configurable"]["checkpoint_id"]

    # Save token → checkpoint mapping for later lookup
    db.save(token=token, thread_id=thread_id, checkpoint_id=checkpoint_id)

    # Send email with approve/reject links
    send_email(
        to      = "manager@company.com",
        subject = f"Approval Needed: {action} (${amount})",
        body    = f"""
        Action : {action}
        Amount : ${amount}

        ✅ Approve : https://yourapp.com/approve?token={token}&decision=approved
        ❌ Reject  : https://yourapp.com/approve?token={token}&decision=rejected
        """
    )

    return {"task_id": thread_id, "status": "pending_approval"}


# ── FastAPI: Email link handler ───────────────────────────────────
@app.get("/approve")
async def handle_email_click(token: str, decision: str):
    record = db.get(token)               # fetch thread_id + checkpoint_id from DB

    # Reconstruct exact checkpoint config
    resume_config = {"configurable": {
        "thread_id":     record["thread_id"],
        "checkpoint_id": record["checkpoint_id"]   # ← pinpoints exact pause point
    }}

    # Inject human decision into state
    graph.update_state(resume_config, {}, resume_as="request_approval")
    # OR directly pass the decision value back to interrupt():
    result = graph.invoke(Command(resume=decision), resume_config)

    return {"status": result["status"], "decision": decision}

---

## Option 2 — REST API Approval

In [ ]:
# ── Start workflow — caller polls for approval ─────────────────────
@app.post("/workflows")
async def create_workflow(user_id: str, action: str, amount: float):
    thread_id = f"user:{user_id}:{uuid.uuid4()}"
    config    = {"configurable": {"thread_id": thread_id}}

    graph.invoke({"action": action, "amount": amount,
                  "status": "pending", "token": ""}, config)

    state = graph.get_state(config)

    # Return task_id — caller stores this and polls /status or sends to approver
    return {
        "task_id":  thread_id,
        "status":   "pending_approval",
        "payload":  state.values           # approver sees this in their UI
    }


# ── Approver submits decision ─────────────────────────────────────
@app.post("/workflows/{task_id}/approve")
async def approve(task_id: str, approved: bool, reviewer: str):
    config = {"configurable": {"thread_id": task_id}}

    state = graph.get_state(config)
    if not state.next:                      # guard: already completed
        return {"error": "Workflow already completed"}

    # Resume graph with human decision
    decision = "approved" if approved else "rejected"
    result   = graph.invoke(Command(resume=decision), config)

    return {"status": result["status"], "reviewer": reviewer}


# ── Check status (caller polls this) ─────────────────────────────
@app.get("/workflows/{task_id}")
async def get_status(task_id: str):
    config = {"configurable": {"thread_id": task_id}}
    state  = graph.get_state(config)

    return {
        "status":    "pending" if state.next else state.values["status"],
        "paused_at": state.next,
        "payload":   state.values
    }

### REST API Caller Flow

In [ ]:
# 1. Trigger workflow
POST /workflows  →  { "task_id": "user:u1:abc", "status": "pending_approval" }

# 2. Poll or show approver the payload
GET  /workflows/user:u1:abc  →  { "status": "pending", "payload": {"action": "Deploy", ...} }

# 3. Approver submits decision
POST /workflows/user:u1:abc/approve  { "approved": true, "reviewer": "Alice" }
                             →  { "status": "approved", "reviewer": "Alice" }

---

## Key Points

In [ ]:
interrupt()         ← pauses graph, exposes data to caller via state.interrupts
Command(resume=x)   ← sends decision back into interrupt(), graph resumes
update_state()      ← alternative: directly set state fields before invoke(None)
checkpoint_id       ← must store in DB for async email flows (server may restart)
PostgresSaver       ← required for external approvals (MemorySaver lost on restart)
state.next == ()    ← completed;  state.next != () ← still paused

| | **Email** | **REST API** |

|---|---|---|

| Approval trigger | Click link in email | POST /approve endpoint |

| Token needed | Yes — maps link → checkpoint | No — caller has task_id |

| `checkpoint_id` in DB | Required | Optional (use thread_id only) |

| Resume method | `Command(resume=decision)` | `Command(resume=decision)` |

| Storage | `PostgresSaver` | `PostgresSaver` |